# ev-flow — quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bertravacca/ev-flow/blob/main/notebooks/ev_flow_quickstart.ipynb)

**ev-flow** generates synthetic plug-in electric-vehicle charging behaviour, grounded in NHTS 2017 travel micro-data and a regional vehicle sales-mix model. You `pip install ev-flow` and `import pev_synth` (one project, two names — the scikit-learn / sklearn convention).

This notebook goes from a clean Colab runtime to a reproducible **fleet load curve** in a few cells.

## 1. Install

In [ ]:
!pip install -q "ev-flow[plotting]"

## 2. Bootstrap the data cache (one-time)

The wheel ships the Python code plus the small SPEECh K=16 parameters, but **not** the cached fleet data — that is built locally from NHTS 2017 micro-data. `ev-flow bootstrap` downloads the NHTS public-use file (~80 MB from ORNL) and builds the cache.

> ⏳ This cell downloads ~80 MB and runs the M2–M7 pipeline, so it takes a few minutes. It only needs to run once per runtime.

In [ ]:
!ev-flow bootstrap --regions bay_area --profile-types residential --n 500

Confirm the environment is ready:

In [ ]:
!ev-flow doctor

## 3. Generate a fleet

Once the cache exists the library API is fast and entirely offline. `generate_profiles` returns a `Fleet` of synthetic EVs; `seed` makes the subset selection reproducible.

In [ ]:
import pev_synth as ps

print(ps.list_regions())
print(ps.list_profile_types())

fleet = ps.generate_profiles('residential', n=300, region='bay_area', seed=42)
fleet

Index into the fleet to get one synthetic EV (a `Profile`):

In [ ]:
prof = fleet[0]
print('archetype :', prof.archetype)
print('battery   :', prof.battery_kwh, 'kWh')
print('max charge:', prof.max_charge_kw, 'kW')

## 4. A fleet load curve

Ask the `Fleet` for its aggregate charging power (kW). The cache lives on a 2001 synthetic calendar, so use 2001 dates.

In [ ]:
load_kw = fleet.aggregate_load('2001-06-01', '2001-06-08', freq='1h')
ax = load_kw.plot(
    figsize=(10, 4),
    ylabel='aggregate charging power (kW)',
    title='Bay Area residential fleet \u2014 one week',
)

## 5. Plug-in count with `pev_synth.plotting`

The optional `pev_synth.plotting` helpers visualise fleet output. `plot_aggregate_load` takes the wide boolean plug-status matrix and plots how many EVs are plugged in over time (connection, not power drawn).

In [ ]:
from pev_synth import plotting

matrix = fleet.plug_status('2001-06-01', '2001-06-08', freq='15min')
ax = plotting.plot_aggregate_load(matrix)

## Next steps

- **Docs:** <https://bertravacca.github.io/ev-flow/>
- **API reference:** every `Fleet` / `Profile` method, the timezone and year-remap rules, and the region registry.
- **Tutorials:** residential load, regional comparison, save/reload a fleet.

Note: `workplace` fleets are fit from a small public EVWatts cohort and carry a documented plug-in-timing caveat (surfaced as a `RuntimeWarning`). V2G, smart-charging and public-charging behaviour are **not** modelled.